In [3]:
import pandas as pd

# Load the CSV from the current directory (adjust the path if needed)
df = pd.read_csv("output.csv")

# Normalize columns just in case
df.columns = df.columns.str.strip()
df["min_char"] = pd.to_numeric(df["min_char"], errors="coerce")

# Filter: TLDs that allow 1-character domains
one_char = df[df["min_char"] == 1].copy()

# Sort for easier scanning
one_char = one_char.sort_values(["tld"])

# 1) Print full table to the console (scrollable in ipynb cell output)
with pd.option_context("display.max_rows", None, "display.max_columns", None, "display.width", 0):
    print(one_char.to_string(index=False))

# 2) Print just the TLDs (one per line) for quick copy/paste
print("\nTLDs allowing 1-character domains:")
for t in one_char["tld"]:
    print(t)


tld  min_char  max_char  registration_price provider
 ac       1.0      63.0                26.0   regery
 af       1.0      63.0                42.0     inwx
 ag       1.0      63.0                79.0   regery
 ai       1.0      63.0                74.0   regery
 am       1.0      63.0                26.0    netim
 ar       1.0      50.0                47.0   regery
 at       1.0      63.0                11.0   regery
 bh       1.0      63.0                53.0   regery
 bo       1.0      63.0                 1.0    netim
 cc       1.0      63.0                 9.0   regery
 cd       1.0      63.0                49.0     inwx
 cf       1.0      63.0                 1.0    netim
 cl       1.0      63.0                13.0   regery
 cm       1.0      63.0                76.0    netim
 cn       1.0      63.0                15.0   regery
 co       1.0      63.0                27.0   regery
 cr       1.0      63.0                93.0    netim
 cv       1.0      63.0                22.0   

In [ ]:
import check_domain
from concurrent.futures import ThreadPoolExecutor, as_completed

# Load RDAP bootstrap once
bootstrap = check_domain.load_bootstrap()

# Load preprocessed TLDs
with open("two_letter_tlds.txt", "r", encoding="utf-8") as f:
    tlds = [line.strip().lstrip(".").lower() for line in f if line.strip()]

base_name = "elijah"
domains = [f"{base_name}.{tld}" for tld in tlds]

def worker(domain: str):
    try:
        return check_domain.check_one(domain, bootstrap)
    except Exception as e:
        return {"domain": domain, "status": "error", "source": "local", "detail": str(e)}

results = []

with ThreadPoolExecutor(max_workers=20) as ex:
    futures = {ex.submit(worker, d): d for d in domains}
    for fut in as_completed(futures):
        res = fut.result()
        results.append(res)
        # Print immediately as the result is ready
        print(f"{res['domain']}: {res['status']} [{res.get('source','')}] - {res.get('detail','')}")

# 🔹 Final sorted output
print("\n--- Final sorted results ---\n")
for r in sorted(results, key=lambda x: x["domain"]):
    print(f"{r['domain']}: {r['status']} [{r.get('source','')}] - {r.get('detail','')}")


elijah.bb: unknown [WHOIS] - No WHOIS server for TLD via whois.iana.org
elijah.at: unknown [WHOIS] - No WHOIS server for TLD via whois.iana.org
elijah.az: unknown [WHOIS] - No WHOIS server for TLD via whois.iana.org
elijah.as: unknown [WHOIS] - No WHOIS server for TLD via whois.iana.org
elijah.aw: unknown [WHOIS] - No WHOIS server for TLD via whois.iana.org
elijah.ao: unknown [WHOIS] - No WHOIS server for TLD via whois.iana.org
elijah.ax: unknown [WHOIS] - No WHOIS server for TLD via whois.iana.org
elijah.an: unknown [WHOIS] - No WHOIS server for TLD via whois.iana.org
elijah.af: unknown [WHOIS] - No WHOIS server for TLD via whois.iana.org
elijah.ba: unknown [WHOIS] - No WHOIS server for TLD via whois.iana.org
elijah.au: unknown [WHOIS] - No WHOIS server for TLD via whois.iana.org
elijah.aq: unknown [WHOIS] - No WHOIS server for TLD via whois.iana.org
elijah.al: unknown [WHOIS] - No WHOIS server for TLD via whois.iana.org
elijah.ai: registered [RDAP] - RDAP 200 domain object returned
e